In [29]:
import pickle, numpy as np, pandas as pd
from sklearn.cluster import AgglomerativeClustering

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import os
from huggingface_hub import InferenceClient

C:\Users\whdgu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#read in complaint embeddings
embed_dict = pickle.load(open('../data/complaint_embeddings.pkl', 'rb'))
#sort by complaint id
sorted_indices = np.argsort(embed_dict['complaint_id'])
embeddings = embed_dict['embeddings'][sorted_indices]
#create df with complaint_id, product, subproduct, company
embeddings_df = pd.DataFrame({
    'complaint_id': np.array(embed_dict['complaint_id'])[sorted_indices],
    'product': np.array(embed_dict['product'])[sorted_indices],
    'subproduct': np.array(embed_dict['sub-product'])[sorted_indices],
    'company': np.array(embed_dict['company'])[sorted_indices],
    'narrative': np.array(embed_dict['narrative'])[sorted_indices],
    'embeddings': list(embeddings)
})
embeddings_df.head(2)

,complaint_id,product,subproduct,company,narrative,embeddings
0,10645738,"Money transfer, virtual currency, or money ser...",Check cashing service,"Paypal Holdings, Inc",I want to withdraw my funds from my Venmo acco...,"[-0.20069551, -0.072827876, 0.0846661, -0.2364..."
1,10647276,Checking or savings account,Savings account,"Block, Inc.",I saw that cashapp recently opened a savings a...,"[-0.04179181, -0.019972146, 0.13940147, -0.038..."


##helper functions

In [22]:

#get embeddings for a specific complaint
def get_embeddings(complaint_id):
    complaint_indices = [i for i, c in enumerate(embed_dict['complaint_id']) if c == complaint_id]
    complaint__indices = sorted(complaint_indices)
    embeddings = embed_dict['embeddings'][complaint_indices]
    return embeddings

def get_experiment_scores(embeddings, cluster_labels):
    '''Get the silhouette score, calinski_harabasz_score, davies_bouldin_score for the given embeddings and distance threshold
    Args:
        embeddings (np.array): The embeddings to cluster.
        cluster_labels (list): The cluster labels for the embeddings.
    Returns:
        dict: A dictionary with the silhouette score, calinski_harabasz_score, davies_bouldin_score.

        silohouette score ranges from -1 to 1, with 1 being the best score.
        calinski_harabasz_score ranges from 0 to infinity, with higher values indicating better clustering.
        davies_bouldin_score ranges from 0 to infinity, with lower values indicating better clustering.
    
    '''
    #get the silhouette score, calinski_harabasz_score, davies_bouldin_score for the given embeddings and distance threshold
    if len(set(cluster_labels)) == 1:
        return {'silhouette': -1,
                'calinski_harabasz': 0,
                'davies_bouldin': float('inf')}
                
    silhouette = silhouette_score(embeddings, cluster_labels)
    calinski_harabasz = calinski_harabasz_score(embeddings, cluster_labels)
    davies_bouldin = davies_bouldin_score(embeddings, cluster_labels)

    return {'silhouette': silhouette,
            'calinski_harabasz': calinski_harabasz,
            'davies_bouldin': davies_bouldin}

def cluster_embeddings(embeeding_df, product, distance_threshold=[1, 5, 10, 15, 20, 25, 30, 35, 40]):
    ''' Cluster the embeddings for a specific product using agglomerative clustering
    Args:

        embeeding_df (pd.DataFrame): The dataframe with embeddings and complaint metadata.
        product (str): The product to cluster.
        distance_threshold (list): The distance thresholds to use for clustering.
    Returns:
        cluster_results (pd.DataFrame): A dataframe with the clustering results for each distance threshold.
    '''
    product_df = embeddings_df.loc[embeddings_df['product'] == product, :]
    if product_df.shape[0] < 2:
        print (f"Not enough complaints for product {product} to cluster.")
        return None, None
    embeddings = np.vstack(embeddings_df.loc[embeddings_df['product'] == product, 'embeddings'].to_numpy())
    cluster_results = pd.DataFrame()
    for dt in distance_threshold:
        clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=dt)
        cluster_labels = clustering.fit_predict(embeddings)
        #write cluster assignments to dataframe with coulmn name 'cluster_label_{dt}'
        product_df[f'cluster_label_{dt}'] = cluster_labels
        num_clusters = len(set(cluster_labels))
        scores = get_experiment_scores(embeddings, cluster_labels)
        avg_cluster_size = len(embeddings) / num_clusters
        cluster_results = pd.concat([cluster_results, pd.DataFrame({
            'distance_threshold': [dt], 
            'num_clusters': [num_clusters],
            'avg_cluster_size': [avg_cluster_size],
            'silhouette': [scores['silhouette']],
            'calinski_harabasz': [scores['calinski_harabasz']],
            'davies_bouldin': [scores['davies_bouldin']]
        })], ignore_index=True)
                                
        print (f"Distance threshold: {dt}, Number of clusters: {num_clusters}")
    return cluster_results, product_df

def cluster_with_gpu(embeddings_subset, distance_threshold):
    ''' Cluster the embeddings with GPU using dbscan    '''
    

In [ ]:
import os
from huggingface_hub import InferenceClient
client = InferenceClient()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "How many 'G's in 'huggingface'?"
        }
    ],
)

print(completion.choices[0].message)


C:\Users\whdgu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatCompletionOutputMessage(role='assistant', content='The word **“huggingface”** contains **3** occurrences of the letter **g** (regardless of case).', reasoning='We need to answer: count the letter \'G\' in the word \'huggingface\'. The word: h u g g i n g f a c e. Lowercase. The question: "How many \'G\'s in \'huggingface\'?" They may be case-sensitive? It asks \'G\'s, uppercase G. In the word there are no uppercase G\'s, but there are lowercase g\'s. Probably they mean letter \'g\' regardless of case. There are three g\'s: positions 3,4,7 (h u g g i n g). So answer: 3.', tool_call_id=None, tool_calls=None)


In [23]:
#size by product
embeddings_df['product'].value_counts()

product
Money transfer, virtual currency, or money service         54312
Checking or savings account                                31893
Credit card                                                21338
Credit reporting or other personal consumer reports        15246
Debt collection                                             4404
Prepaid card                                                1751
Vehicle loan or lease                                       1669
Mortgage                                                    1243
Payday loan, title loan, personal loan, or advance loan      415
Debt or credit management                                    205
Student loan                                                  36
Name: count, dtype: int64

In [25]:
#try clustering each product's embeddings separately

products = embeddings_df['product'].unique()
product_summary = pd.DataFrame()
embeddings_df_clustered = pd.DataFrame()
for product in products[[-1]]:
    print(f"Clustering product: {product}")
    cluster_results, product_df = cluster_embeddings(embeddings_df, product, distance_threshold=[1, 5, 10, 15, 20, 25, 30, 35, 40])
    embeddings_df_clustered = pd.concat([embeddings_df_clustered, product_df], ignore_index=True)
    print(f'cluster results for product {product}:')
    print(cluster_results)
    cluster_results['product'] = product
    product_summary = pd.concat([product_summary, cluster_results], ignore_index=True)
product_summary.to_csv('../data/product_clustering_summary.csv', index=False)
product_summary


    

Clustering product: Debt or credit management
Distance threshold: 1, Number of clusters: 203
Distance threshold: 5, Number of clusters: 30
Distance threshold: 10, Number of clusters: 5
Distance threshold: 15, Number of clusters: 3
Distance threshold: 20, Number of clusters: 2
Distance threshold: 25, Number of clusters: 2
Distance threshold: 30, Number of clusters: 1
Distance threshold: 35, Number of clusters: 1
Distance threshold: 40, Number of clusters: 1
cluster results for product Debt or credit management:
   distance_threshold  num_clusters  avg_cluster_size  silhouette  \
0                   1           203          1.009852    0.015015   
1                   5            30          6.833333    0.062146   
2                  10             5         41.000000    0.077671   
3                  15             3         68.333333    0.073917   
4                  20             2        102.500000    0.165270   
5                  25             2        102.500000    0.165270   
6

C:\Users\whdgu\AppData\Local\Temp\ipykernel_22536\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_22536\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_22536\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

,distance_threshold,num_clusters,avg_cluster_size,silhouette,calinski_harabasz,davies_bouldin,product
0,1,203,1.009852,0.015015,91.477020,0.076784,Debt or credit management
1,5,30,6.833333,0.062146,7.393061,2.021129,Debt or credit management
2,10,5,41.000000,0.077671,22.292389,2.868949,Debt or credit management
3,15,3,68.333333,0.073917,33.317371,2.472456,Debt or credit management
4,20,2,102.500000,0.165270,46.009750,2.056710,Debt or credit management
5,25,2,102.500000,0.165270,46.009750,2.056710,Debt or credit management
6,30,1,205.000000,-1.000000,0.000000,inf,Debt or credit management
7,35,1,205.000000,-1.000000,0.000000,inf,Debt or credit management
8,40,1,205.000000,-1.000000,0.000000,inf,Debt or credit management


In [26]:
embeddings_df_clustered

,complaint_id,product,subproduct,company,narrative,embeddings,cluster_label_1,cluster_label_5,cluster_label_10,cluster_label_15,cluster_label_20,cluster_label_25,cluster_label_30,cluster_label_35,cluster_label_40
0,10720566,Debt or credit management,Credit repair services,AMERICAN EXPRESS COMPANY,I agreed a settlement with American Express an...,"[0.086922474, -0.09507114, 0.22637866, 0.22611...",177,5,0,0,1,1,0,0,0
1,10912151,Debt or credit management,Debt settlement,AMERICAN EXPRESS COMPANY,We were eligible for the FXXXX XXXX XXXX XXXX ...,"[-0.052172467, 0.06367002, 0.14233169, 0.13692...",189,15,2,2,0,0,0,0,0
2,10953735,Debt or credit management,Mortgage modification or foreclosure avoidance,NAVY FEDERAL CREDIT UNION,Over 5 months of hardship and harassment. \nAp...,"[-0.035568345, 0.25348818, 0.109614685, -0.030...",155,26,4,0,1,1,0,0,0
3,10995750,Debt or credit management,Debt settlement,"Paypal Holdings, Inc",Pay pal allowance of unauthorized payments. No...,"[-0.24532488, 0.1985969, 0.23773563, -0.099170...",115,28,0,0,1,1,0,0,0
4,10996183,Debt or credit management,Debt settlement,SYNCHRONY FINANCIAL,I never received any mail because I had a XXXX...,"[-0.13533992, -0.055381577, 0.6002768, 0.02616...",131,14,4,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,15431915,Debt or credit management,Credit repair services,WELLS FARGO & COMPANY,I applied for a personal loan to repay my cred...,"[-0.2564726, -0.1275893, 0.22124344, 0.0722324...",13,4,4,0,1,1,0,0,0
201,15481146,Debt or credit management,Credit repair services,"BANK OF AMERICA, NATIONAL ASSOCIATION",My ex husband is using a credit card that is X...,"[-0.10727175, -0.19191548, 0.28368923, -0.1621...",3,14,4,0,1,1,0,0,0
202,15615130,Debt or credit management,Debt settlement,"CITIBANK, N.A.","To the Consumer Financial Protection Bureau, I...","[-0.2940723, 0.110055685, 0.020373518, 0.01175...",1,8,2,2,0,0,0,0,0
203,15787998,Debt or credit management,Mortgage modification or foreclosure avoidance,WELLS FARGO & COMPANY,Wells Fargo and Hud Servicer XXXX have a withh...,"[-0.330733, 0.042372834, 0.14269748, 0.0155245...",6,15,2,2,0,0,0,0,0


In [ ]:
def get_llm_response(prompt):
    client = InferenceClient()

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )
    return completion.choices[0].message['content']
def get_llm__cluster_name(cluster_narratives):
    prompt = f"Given the following customer complaint narratives, provide a concise name that captures the common theme among them. Narratives: {cluster_narratives}"
    response = get_llm_response(prompt)
    return response

def name_clusters(embeddings_df, cluster_col='cluster_label',product_subset=None):
    '''
    Name clusters using LLM based on a smaple of narratives from each each product and cluster label.
    Args:
        embeddings_df (pd.DataFrame): DataFrame containing complaint narratives and cluster labels.
        cluster_col (str): Column name in embeddings_df that contains cluster labels.
    Returns:
        dict: A dictionary mapping products and cluster labels to their assigned names.

    '''
    #check that there are less than 50 clusters
    
    
    cluster_names = {}
    if product_subset is not None:
        embeddings_df = embeddings_df.loc[embeddings_df['product'].isin(product_subset), :]
    else:
        products = embeddings_df['product'].unique()
    for product in products:
        product_df = embeddings_df.loc[embeddings_df['product'] == product, :]
        cluster_labels = product_df[cluster_col].unique()
        for cluster_label in cluster_labels:
            cluster_df = product_df.loc[product_df[cluster_col] == cluster_label, :]
            #sample up to 5 narratives from the cluster
            sample_narratives = cluster_df['narrative'].sample(n=min( len(cluster_df), 7), random_state=42).tolist()
            cluster_name = get_llm__cluster_name(sample_narratives)
            cluster_names[(product, cluster_label)] = cluster_name
            print(f"Product: {product}, Cluster: {cluster_label}, Name: {cluster_name}")
    return cluster_names
cluster_names = name_clusters(embeddings_df_clustered, cluster_col='cluster_label_5')

#add cluster names to embeddings_df


Product: Debt or credit management, Cluster: 5, Name: **Credit Report Dispute / Inaccurate Reporting**
Product: Debt or credit management, Cluster: 15, Name: **“FCRA‑Related Credit Reporting & Consumer Credit Dispute Complaints”**
Product: Debt or credit management, Cluster: 26, Name: **“Identity Theft & Financial Fraud Complaints”**
Product: Debt or credit management, Cluster: 28, Name: **Payment Platform Dispute & Refund Issues**
Product: Debt or credit management, Cluster: 14, Name: **Credit Dispute & Fraud Complaints**
Product: Debt or credit management, Cluster: 10, Name: **Misleading Credit Reporting & Billing Errors**
Product: Debt or credit management, Cluster: 8, Name: **Consumer Banking & Credit Dispute Complaints**
Product: Debt or credit management, Cluster: 22, Name: **Financial Service Disputes & Fraud**
Product: Debt or credit management, Cluster: 4, Name: **Predatory Financial Practices & Harassment**
Product: Debt or credit management, Cluster: 3, Name: **Identity Thef